# Advanced modeling: harmonic analysis of pulsar angular correlations

We reproduce (some) results from the NANOGrav 15-year [harmonic analysis paper](https://arxiv.org/pdf/2411.13472). Following the notation of the paper, we perform parameter estimation for the $\text{HA}^\gamma(c_2, c_3)$ model.

This is non-trivial for Prometheus because the `CommonSpectralModel` object assumes the correlation matrix is fixed. Harmonic analysis requires the inter-pulsar correlations vary. We must implement this with the more general `SpectralModel` object.

In [ ]:
# to load/save objects
import pickle

# for plotting
import numpy as np
import math
import matplotlib.pyplot as plt
import corner

# for sampling
import jax
import jax.numpy as jnp
import jax.random as jr
import numpyro

import sys
sys.path.append('../..')

# prometheus objects
from prometheus.spectral_models import SpectralModel
from prometheus import spectra
from prometheus.pta_model import PTAModel

%load_ext autoreload
%autoreload 2

In [ ]:
# check that we are running on a GPU
# this should print something like "CudaDevice(id=...)"
print(jax.devices())

In [ ]:
# load a previously constructed data object (see gwb_pe.ipynb)
with open('../../data/NG15/data.pkl', 'rb') as fp:
    dataset = pickle.load(fp)

## Performing Harmonic Analysis

The `CommonSpectralModel` class provides a convenient way to construct stocahstic gravitational wave background models using a user-supplied `get_phi_diag_func` and pulsar correlation matrix. However, this correlation is considered fixed, and cannot be parameterized and varied in a Bayesian sampling scheme.

---

### General spectral models

To support parameterized correlation matrices, Prometheus provides the more general `SpectralModel` class.

Unlike `IndependentSpectralModel`, which vectorizes a `get_phi_diag_func` across pulsars, and `CommonSpectralModel`, which applies a `get_phi_diag_func` commonly to all pulsars under some correlation pattern, `SpectralModel` requires a `get_phi_cube_func`. This function takes a 1d-array of parameters and a 1d-array of frequencies as input, and outputs a $(2N_f, N_p, N_p)$ array where $N_f$ is the number of frequency bins and $N_p$ is the number of pulsars. The $(i, j, k)$ element of this output array is the covariance of the $i^\text{th}$ Fourier component between pulsars $j$ and $k$.

The custom `SpectralModel` object allows for more flexible modeling of Gaussian processes.

### Model composition rules in Prometheus

The construction of a `prometheus.pta_model.PTAModel` requires **either**:

- a pair of spectral models:
  - `IndependentSpectralModel`: usually for pulsar noise
  - `CommonSpectralModel`: usually for a GWB or common process

**or**

- a single `SpectralModel`

If you supply a custom `SpectralModel`, your `get_phi_cube_func` **must include all desired contributions** to Gaussian processes, e.g. pulsar noise and GWB contributions.

---

### Example model

We will implement and conduct parameter estimation for a model where:

- all pulsar noise is modeled with a power law
- the GWB uses a power law model with $N_f=14$ frequency bins
- the GWB inter-pulsar correlation pattern is parameterized using a Legendre expansion

Following the [harmonic analysis paper](https://arxiv.org/pdf/2411.13472), we parameterize the angular correlations by (Eq. 9)
\begin{equation}
  \Gamma_{ab}^{\ell_\text{max}} = (1-\delta_{ab})\sum_{\ell=2}^{\ell_\text{max}}c_\ell P_\ell(\cos\theta_{ab})+\delta_{ab}
\end{equation}
where $P_\ell$ denotes the $\ell^\text{th}$ Legendre polynomial. We expand up the correlation pattern up to $\ell_\text{max}=3$.

Throughout this notebook, we order parameters as $(c_2, c_3, \log_{10}A_\text{GWB}, \gamma_\text{GWB}, \log_{10}A_1, \gamma_1,...)$.

In [ ]:
# some constants for the analysis
npsrs = dataset.npsrs
nfreqs = dataset.nfreqs

# indices of pulsars
ii_psrs = jnp.arange(npsrs)

# number of frequency bins for GWB
nfreq_GWB = 14

# number of parameters
nLegendre = 2
nparams_per_power_law = 2
nparams_gwb = nLegendre + nparams_per_power_law
nparams_psr = npsrs * nparams_per_power_law

# store cos(angular separation)
cos_ang_separation = np.clip(np.dot(dataset.psrpos, dataset.psrpos.T), -1, 1)

from scipy.special import legendre
Legendres = jnp.array([legendre(i + 2)(cos_ang_separation)
                       for i in range(nLegendre)])

## Build `get_phi_cube_func`

In [ ]:
# GWB contribution to spectrum
def gwb_get_phi_cube(gwb_params, freqs):

    # unpack parameters
    lnLegendre_weights, power_law_params = (gwb_params[:nLegendre], gwb_params[nLegendre:])

    # to help the sampler, we're going to sample in log-Legendre-coefficeints
    # convert from natural-log to proper Legendre weights
    Legendre_weights = jnp.exp(lnLegendre_weights)

    # the GWB is a power law spectrum
    gwb_phi_diag = spectra.power_law(power_law_params, freqs)

    # zero-out frequencies above Nf=14
    zeros = jnp.zeros((2 * (dataset.nfreqs - nfreq_GWB)))
    gwb_phi_diag = jnp.concatenate((gwb_phi_diag[:2 * nfreq_GWB], zeros))

    # construct correlation matrix
    Legendre_sum = jnp.sum(Legendre_weights[:, None, None] * Legendres, axis=0)
    correlation_matrix = (1 - jnp.eye(npsrs)) * Legendre_sum + jnp.eye(npsrs)

    # project spectral model over correlation pattern
    phi_cube = gwb_phi_diag[:, None, None] * correlation_matrix[None, :, :]

    return phi_cube

In [ ]:
# the pulsar noise model
def psr_noise_get_phi_cube(psr_params, freqs):

    # initialize empty covariance matrix to populate
    phi_cube = jnp.zeros((freqs.shape[0], npsrs, npsrs))

    # reshape pulsar power law parameters
    nparams_per_power_law = 2   # (amplitude, spectral index)
    psr_params_stacked = psr_params.reshape((npsrs, nparams_per_power_law))

    # get power spectrum
    phi_diags_per_psr = jax.vmap(lambda x: spectra.power_law(x, freqs))(psr_params_stacked).T

    # fill in covariance matrix
    phi_cube = phi_cube.at[:, ii_psrs, ii_psrs].add(phi_diags_per_psr)

    return phi_cube

In [ ]:
# construct total get_phi_cube_func
# this needs to include all contributions to Gaussian processes
def get_phi_diag_func(params, freqs):

    # unpack parameters
    gwb_params, psr_params = (params[:nparams_gwb], params[nparams_gwb:])

    return (gwb_get_phi_cube(gwb_params, freqs)
            + psr_noise_get_phi_cube(psr_params, freqs))

## Construct Spectral Model

In [ ]:
# GWB parameter bounds
lnLegendre_weight_mins = jnp.ones(nLegendre) * (-10.)
lnLegendre_weight_maxs = jnp.zeros(nLegendre)
gwb_power_law_mins = jnp.array([-20., 0.])
gwb_power_law_maxs = jnp.array([-10., 7.])

# parameter bounds for pulsar noise model
psr_power_law_param_mins = jnp.array([[-20., 0.]
                                       for _ in range(npsrs)]).flatten()
psr_power_law_param_maxs = jnp.array([[-12., 7.]
                                       for _ in range(npsrs)]).flatten()

# concatenate into bounds for all parameters
param_mins = jnp.concatenate((lnLegendre_weight_mins,
                              gwb_power_law_mins,
                              psr_power_law_param_mins))
param_maxs = jnp.concatenate((lnLegendre_weight_maxs,
                              gwb_power_law_maxs,
                              psr_power_law_param_maxs))
parameter_bounds = jnp.array([param_mins, param_maxs]).T

### Fixing the prior

Notice we don't sample the coefficients of the Legendre expansion, $c_\ell$, directly. Rather we sample $\ln c_\ell$ and Prometheus will automatically implement a uniform prior for $\ln c_\ell$ (log-uniform in $c_\ell$). However, we desire a prior which is uniform in $c_\ell$!

To fix this, we add the log-determinant of the Jacobian of this reparameterization below.

In [ ]:
def add_ln_factor(params):
    lnLegendre_weights = params[:nLegendre]
    return jnp.sum(lnLegendre_weights)

In [ ]:
# build custom spectral model
harmonic_analysis_spectral_model = SpectralModel(name='gwb_&_psr_params',
                                         parameter_bounds=parameter_bounds,
                                         data=dataset,
                                         get_phi_cube_func=get_phi_diag_func)

# now we can build a PTAModel from the SpectralModel
pta_model = PTAModel(spectral_model=harmonic_analysis_spectral_model,
                     add_ln_factor=add_ln_factor,   # to fix the prior
                     )

## Sample as usual

In [ ]:
# get probabilistic sampling model from the PTA model
sampling_model = pta_model.sampling_model

# build the NumPyro NUTS kernel
nuts_kernel = numpyro.infer.NUTS(model=sampling_model)

# specify MCMC attributes
mcmc = numpyro.infer.MCMC(sampler=nuts_kernel,
                          num_warmup=1000,
                          num_samples=5000)

# seed to start sampling
# if you see step size < 1e-5, change this seed and try again!
seed = 63445

# run MCMC and get samples
mcmc.run(jr.PRNGKey(seed))
samples = mcmc.get_samples()

## Post-processing

Show recovery of GWB and inter-pulsar correlation.

In [ ]:
# extract GWB parameters
gwb_samples = np.array(samples['gwb_&_psr_params'])[:, :nparams_gwb]

# transform from log- back to Legendre coefficients
gwb_samples[:, :nLegendre] = np.exp(gwb_samples[:, :nLegendre])

# make labels
Legendre_weight_labels = np.array([rf'$c_{{{2 + i}}}$'
                                   for i in range(nLegendre)])
gwb_spectrum_labels = np.array([r'$\log_{10}A_\text{GWB}$',
                                r'$\gamma_\text{GWB}$'])
labels = np.concatenate((Legendre_weight_labels, gwb_spectrum_labels))

# HD predicts Legendre weights
Legendre_HD_weights = np.array([3 / 2 * (2 * l + 1) * math.factorial(l - 2) / math.factorial(l + 2)
                                for l in range(2, 2 + nLegendre)])
power_law_reference_params = np.array([None] * nparams_per_power_law)
reference_params = np.concatenate((Legendre_HD_weights, power_law_reference_params))

# make corner plot
fig = corner.corner(data=gwb_samples,
                    labels=labels,
                    truths=reference_params,
                    bins=20,
                    hist_kwargs={'density': True},
                    label_kwargs={'fontsize': 16})

In [ ]:
# Plot recovered correlation pattern

# angular separations to plot over
theta_degree = np.linspace(0.1, 179.9, 100)
theta_rad = theta_degree * np.pi / 180
cos_theta = np.cos(theta_rad)

# HD curve
beta = (1 - np.cos(theta_rad)) / 2
HD_curve = 1/2 - 1/4*beta + 3/2*beta*np.log(beta)

# approximate HD curve from expansion
legendres_cos_theta = np.array([legendre(i + 2)(cos_theta)
                                for i in range(nLegendre)])
HD_curve_exp = np.sum(Legendre_HD_weights[:, None] * legendres_cos_theta, axis=0)

# samples of correlation pattern
Legendre_weights_samples = np.exp(np.array(samples['gwb_&_psr_params'])[:, :nLegendre])
Legendre_sums = np.sum(Legendre_weights_samples[:, :, None] * legendres_cos_theta[None, :, :],
                       axis=1)

# get 1\sigma and 2\sigma credible regions
# note we compute these credible regions in function space,
# rather than parameter space
percent2p5, percent16, percent84, percent97p5 = np.percentile(
    Legendre_sums, [2.5, 16, 84, 97.5], axis=0
)

# make figure
plt.figure(figsize=(6, 4))
plt.fill_between(theta_degree, percent2p5, percent97p5, color='C0', alpha=0.3)
plt.fill_between(theta_degree, percent16, percent84, color='C0', alpha=0.5)
plt.plot(theta_degree, HD_curve, ls='--', color='k', lw=2, label='HD correlation')
plt.plot(theta_degree, HD_curve_exp, ls='--', color='C3',
         label=r'HD correlation ($\ell_\text{max}=$' + f'{nLegendre+1})', lw=2, alpha=0.7)
plt.plot([], [], lw=8, color='C0', alpha=0.5,
         label=(r'$\pm 1\sigma$, $\pm 2\sigma$ credible regions'
                + '\n' + rf'(varying $\ell={list(range(2, 2 + nLegendre))}$)'))
plt.axhline(0, color='k', alpha=0.6)
plt.legend(loc='upper right')
plt.xlabel('Pulsar-pair angular separation (degrees)')
plt.ylabel('angular correlation function')
plt.title('NG15 Harmonic Analysis')
plt.show()